Python Script for train and inference of an LGN and an FFN  

Inference and training functions based on /experiments/main.py from https://github.com/Felix-Petersen/difflogic
and Filipa Trindade's codes in "BOOLEAN ALGEBRA: FROM DIGITAL CIRCUITS TO DEEP LEARNING APPLICATIONS © 2024"
 
Author: Mahmoud Yassin © 2026

 



**Color reminder (black background)**

LaTeX (MathJax) needs `$...$`:

- `$\color{#4FC3F7}{\text{Cyan}}$`
- `$\color{#2DD4BF}{\text{Teal}}$`
- `$\color{#A3E635}{\text{Green}}$`
- `$\color{#FACC15}{\text{Yellow}}$`
- `$\color{#FB923C}{\text{Orange}}$`
- `$\color{#F472B6}{\text{Pink}}$`
- `$\color{#C084FC}{\text{Violet}}$`
- `$\color{#E5E7EB}{\text{Light gray}}$`

HTML works too:
`<span style="color: #4FC3F7;">Cyan text</span>`


### Import libraries

In [ ]:
import os
import torch

torch_pkg_dir = os.path.dirname(torch.__file__)          # ...\site-packages\torch
torch_lib_dir = os.path.join(torch_pkg_dir, "lib")       # ...\site-packages\torch\lib

print("torch_pkg_dir =", torch_pkg_dir)
print("torch_lib_dir =", torch_lib_dir, "exists =", os.path.isdir(torch_lib_dir))

os.add_dll_directory(torch_lib_dir)
os.add_dll_directory(r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.1\bin")

import difflogic_cuda
print("difflogic_cuda loaded OK")
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))
print("difflogic_cuda:", difflogic_cuda)

In [9]:
# Import Libraries
import pandas as pd
import numpy as np
import plotly.graph_objects as go

import math
import random
import os
import sys
import time
import datetime
import json
import pickle
import statistics
from pathlib import Path

import torch
import torchvision
from tqdm import tqdm
from copy import deepcopy

import torchvision.transforms as transforms
from PIL import Image
from torchvision.datasets import MNIST
from torch.utils.data import random_split

# Force this notebook to use the clean local difflogic package from the repo.
repo_difflogic = Path('/home/sci/Documents/Github_projects/difflogic')
if str(repo_difflogic) not in sys.path:
    sys.path.insert(0, str(repo_difflogic))

#from results_json import ResultsJSON
import difflogic
print('Using difflogic from:', difflogic.__file__)
from difflogic import LogicLayer, GroupSum, CompiledLogicNet, PackBitsTensor
#if torch.cuda.is_available():
 #   import difflogic_cuda

Using difflogic from: /home/sci/miniconda3/envs/ppddlgn-difflogic-pip117/lib/python3.10/site-packages/difflogic/__init__.py


In [ ]:
import torch
print(torch.__version__, torch.version.cuda, torch.cuda.is_available())
if torch.cuda.is_available():
    import difflogic_cuda
    print(difflogic_cuda)
else:
    print('CUDA not available; skipping difflogic_cuda import')


## Balance dataset and create directory structure
### Don't run this cell if you already have your dataset ready
## $$\color{#C084FC}{\text{Don't run this cell if you already have your dataset ready}}$$

### Dictionary of model parameters (LGN)

In [10]:
# Define dictonary of model parameters
model_size = "medium"  # "small" or "normal"

paper_cfg = {
    "small":  {"num_neurons": 4000,  "tau": 3.0,"num_layers": 2},       # 
    "Large": {"num_neurons": 8000,  "tau": 10.0,"num_layers": 6},     # 
    "medium": {"num_neurons": 6000,  "tau": 5.0, "num_layers": 4},
}

args = {
    "id": 10,
    "tau": paper_cfg[model_size]["tau"],
    "batch_size": 100,
    "learning_rate": 0.01,
    "epochs": 200,
    "patience": 10,
    "num_neurons": paper_cfg[model_size]["num_neurons"],
    "num_layers": paper_cfg[model_size]["num_layers"],
    "img_size": 28,
    "seed": 952,
    "valid_set_size": 0.1,
    "pixel_mode": "threshold_05",  # "multi_threshold" or "threshold_05"
    "num_niveis": 3,                  # used only when pixel_mode == "multi_threshold"
}



### Define seed

In [11]:
# Define Seed
torch.manual_seed(args['seed'])
random.seed(args['seed'])
np.random.seed(args['seed'])

In [12]:
def make_mnist_transform(mode, num_niveis=3):
    if mode == "multi_threshold":
        # output: [num_niveis, 28, 28]
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Lambda(
                lambda x: torch.cat(
                    [(x > ((i + 1) / (float(num_niveis) + 1))).float() for i in range(num_niveis)],
                    dim=0
                )
            ),
        ])
    if mode == "threshold_05":
        # output: [1, 28, 28]
        return transforms.Compose([
            transforms.ToTensor(),
            transforms.Lambda(lambda x: (x > 0.5).float()),
        ])
    raise ValueError(f"Unknown pixel_mode: {mode}")

In [13]:

mnist_tf = make_mnist_transform(args["pixel_mode"], args["num_niveis"])

full_train = MNIST(root="./data-mnist", train=True, download=True, transform=mnist_tf)
test_images = MNIST(root="./data-mnist", train=False, download=True, transform=mnist_tf)

train_size = int((1 - args["valid_set_size"]) * len(full_train))
val_size = len(full_train) - train_size
train_images, val_images = random_split(
    full_train, [train_size, val_size],
    generator=torch.Generator().manual_seed(args["seed"])
)
print(f"Train images: {len(train_images)}")
print(f"Val images:   {len(val_images)}")
print(f"Test images:  {len(test_images)}")
print(f"Total check (train+val): {len(train_images) + len(val_images)} / {len(full_train)}")


Train images: 54000
Val images:   6000
Test images:  10000
Total check (train+val): 60000 / 60000


### Folder for training iterations

In [65]:
# Create a folder for each training iteration
# Define timestamp for the folder's name
time_string = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
# Folder for LGN trainings
path = 'trained_models' + '/' + time_string
if not os.path.exists(path):
    os.makedirs(path)

In [66]:
# Creates the file log_args.txt and saves our args. dictionary
with open(os.path.join(path, 'log_args.txt'), 'w') as file:
    file.write(json.dumps(args))

### Loading the pictures

In [14]:
# Transform the folders in Dataloaders
train_loader = torch.utils.data.DataLoader(train_images, batch_size=args['batch_size'], shuffle=True, drop_last=True, num_workers=0)
val_loader   = torch.utils.data.DataLoader(val_images,   batch_size=args['batch_size'], shuffle=False, drop_last=False, num_workers=0)
test_loader  = torch.utils.data.DataLoader(test_images,  batch_size=args['batch_size'], shuffle=False, drop_last=False, num_workers=0)

#results = ResultsJSON(eid=args['id'], path='./results/')

In [56]:
#just to check the dataloader
x, y = next(iter(train_loader))
print(f"Batch shape - Images: {x.shape}, Labels: {y.shape}")
print(f"Sample label: {y}")
num_batches = len(train_loader)
print(f"Number of batches: {num_batches}")
len(train_loader)

Batch shape - Images: torch.Size([100, 1, 28, 28]), Labels: torch.Size([100])
Sample label: tensor([6, 4, 6, 6, 1, 2, 1, 8, 0, 1, 0, 0, 7, 0, 6, 7, 0, 5, 1, 5, 3, 9, 8, 6,
        3, 8, 8, 1, 8, 2, 8, 8, 2, 6, 1, 2, 5, 8, 5, 2, 1, 0, 1, 1, 9, 4, 2, 7,
        0, 3, 3, 9, 3, 4, 7, 5, 2, 1, 7, 4, 8, 4, 0, 2, 1, 9, 4, 9, 1, 6, 5, 8,
        7, 0, 0, 2, 4, 7, 5, 0, 8, 4, 9, 2, 6, 3, 1, 5, 5, 1, 0, 6, 6, 9, 5, 1,
        1, 2, 6, 9])
Number of batches: 540


540

### Define the functions for train and inference

In [4]:
# Define the functions for train and inference
def train(model, x, y, loss_fn, optimizer):
    x = model(x)
    loss = loss_fn(x, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

def _get_model_device(model):
    try:
        return next(model.parameters()).device
    except StopIteration:
        return torch.device("cpu")

def eval(model, loader, mode, device=None):
    """
    Computes mean accuracy over a loader.
    mode=False -> hard/discrete eval behavior
    mode=True  -> soft behavior
    """
    device = torch.device(device) if device is not None else _get_model_device(model)
    orig_mode = model.training
    with torch.no_grad():
        model.train(mode=mode)
        res = np.mean([
            (model(x.to(device).round()).argmax(-1) == y.to(device))
            .to(torch.float32).mean().item()
            for x, y in loader
        ])
        model.train(mode=orig_mode)
    return float(res)

def packbits_eval(model, loader, device=None):
    device = torch.device(device) if device is not None else _get_model_device(model)
    orig_mode = model.training
    with torch.no_grad():
        model.eval()
        res = np.mean([
            (model(PackBitsTensor(x.to(device).reshape(x.shape[0], -1).round().bool())).argmax(-1) == y.to(device))
            .to(torch.float32).mean().item()
            for x, y in loader
        ])
        model.train(mode=orig_mode)
    return float(res)


### LGN training

In [70]:
#####################
### LGN TRAINING  ###
#####################
llkw = dict(grad_factor=1., connections='unique')

# Input dimension depends on pixel representation
if args["pixel_mode"] == "multi_threshold":
    in_dim = args["num_niveis"] * args["img_size"] * args["img_size"]
elif args["pixel_mode"] == "threshold_05":
    in_dim = args["img_size"] * args["img_size"]
else:
    raise ValueError(f"Unknown pixel_mode: {args['pixel_mode']}")

# MNIST is 10 classes
class_count = 10

# choose device depending on availability (select before creating layers)
device = "cuda" if torch.cuda.is_available() else "cpu"

logic_layers = []
arch = 'randomly_connected'
k = args['num_neurons']
l = args['num_layers']

logic_layers.append(torch.nn.Flatten())
logic_layers.append(LogicLayer(in_dim=in_dim, out_dim=k, device=device, **llkw))
for _ in range(l - 1):
    logic_layers.append(LogicLayer(in_dim=k, out_dim=k, device=device, **llkw))

model = torch.nn.Sequential(
    *logic_layers,
    GroupSum(class_count, args['tau'])
)

total_num_neurons = sum(layer.num_neurons for layer in logic_layers[1:])
total_num_weights = sum(layer.num_weights for layer in logic_layers[1:])

print(f'total_num_neurons={total_num_neurons}')
print(f'total_num_weights={total_num_weights}')

# move model to selected device and set implementation flag
model = model.to(device)
model.implementation = "cuda" if device == "cuda" else "cpu"
print(model)
print(f'Using device: {device}')

loss_fn = torch.nn.CrossEntropyLoss()
optim = torch.optim.Adam(model.parameters(), lr=args['learning_rate'])

pytorch_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('Total trainable params:' + str(pytorch_total_params))


total_num_neurons=24000
total_num_weights=24000
Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): LogicLayer(784, 6000, train)
  (2): LogicLayer(6000, 6000, train)
  (3): LogicLayer(6000, 6000, train)
  (4): LogicLayer(6000, 6000, train)
  (5): GroupSum(k=10, tau=5.0)
)
Using device: cpu
Total trainable params:384000


In [71]:
# Define necessary variables for training
val_anterior = -1.0
val_counter = 0
val_list, train_list, loss_list, seconds_list = [], [], [], []
cumu_training_seconds = 0
best_epoch = 0
best_model = deepcopy(model)  # safety: always defined

# ensure device is set consistently (use earlier selection if present)
device = "cuda" if torch.cuda.is_available() else "cpu"

for epoch in range(args['epochs']):
    start_time = time.time()
    print('Epoch: ' + str(epoch))

    for i, (x, y) in tqdm(enumerate(train_loader), desc='Epoch', total=len(train_loader)):
        x = x.to(torch.float32).to(device)   # float32 (not float64)
        y = y.to(device)
        loss = train(model, x, y, loss_fn, optim)

    cumu_training_seconds += (time.time() - start_time)
    seconds_list.append(round(cumu_training_seconds, 0))

    print('Computing validation accuracy...')
    val_accuracy_eval_mode = eval(model, val_loader, mode=False, device=device)
    train_accuracy_eval_mode = eval(model, train_loader, mode=False, device=device)

    val_list.append(val_accuracy_eval_mode)
    train_list.append(train_accuracy_eval_mode)
    loss_list.append(loss)

    print('Loss: ' + str(np.round(loss, 4)))
    print('Val Acc: ' + str(np.round(val_accuracy_eval_mode, 4)))
    print('Train Acc: ' + str(np.round(train_accuracy_eval_mode, 4)))

    if (val_accuracy_eval_mode <= val_anterior) or (epoch == args['epochs'] - 1):
        val_counter += 1
        if (val_counter > args['patience']) or (epoch == args['epochs'] - 1):
            pickle.dump(best_model, open(os.path.join(path, f'epoch-{best_epoch}'), 'wb'))
            with open(os.path.join(path, 'log_seconds.txt'), 'w') as file:
                file.write(json.dumps(seconds_list))
            with open(os.path.join(path, 'log_val.txt'), 'w') as file:
                file.write(json.dumps(val_list))
            with open(os.path.join(path, 'log_train.txt'), 'w') as file:
                file.write(json.dumps(train_list))
            break
    else:
        val_counter = 0
        best_model = deepcopy(model)
        best_epoch = epoch

    val_anterior = max(val_anterior, val_accuracy_eval_mode)

model = pickle.load(open(os.path.join(path, f'epoch-{best_epoch}'), 'rb'))
test_accuracy_eval_mode = eval(model, test_loader, mode=False, device=device)
print('Test Acc: ' + str(np.round(test_accuracy_eval_mode, 4)))


Epoch: 0


Epoch: 100%|██████████| 540/540 [02:24<00:00,  3.73it/s]


Computing validation accuracy...
Loss: 0.3079
Val Acc: 0.9003
Train Acc: 0.8973
Epoch: 1


Epoch: 100%|██████████| 540/540 [02:23<00:00,  3.76it/s]


Computing validation accuracy...
Loss: 0.1873
Val Acc: 0.899
Train Acc: 0.8985
Epoch: 2


...

Computing validation accuracy...
Loss: 0.0182
Val Acc: 0.96
Train Acc: 0.9956
Epoch: 78


Epoch: 100%|██████████| 540/540 [01:50<00:00,  4.88it/s]


Computing validation accuracy...
Loss: 0.0217
Val Acc: 0.9617
Train Acc: 0.995
Epoch: 79


Epoch: 100%|██████████| 540/540 [01:51<00:00,  4.85it/s]


Computing validation accuracy...
Loss: 0.0062
Val Acc: 0.9625
Train Acc: 0.9953
Test Acc: 0.9611


In [25]:
# Code to load the best model from the folder and evaluate (before/after discretization)
import os
import pickle
import time
import torch

#model_path = r"trained_models/20260207_121201/epoch-38"   # safe
model_path = r"trained_models/20260309_224007/Model_medium_epoch-68"
# or: model_path = r"trained_models\20260207_121201\epoch-38"

print("Exists:", os.path.exists(model_path))
#runtime_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
runtime_device = "cpu"

best_lgn_model = pickle.load(open(model_path, "rb")).to(runtime_device)
print(f"Using runtime device: {runtime_device}")
def timed_eval(model, loader, mode, device=None):
    device = torch.device(device) if device is not None else _get_model_device(model)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    acc = eval(model, loader, mode=mode, device=device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t1 = time.perf_counter()
    return acc, (t1 - t0)

# Before discretization: keep soft gates (train mode)
acc_soft, time_soft = timed_eval(best_lgn_model, test_loader, mode=True)

# After discretization: use hard gates (eval mode)
acc_hard, time_hard = timed_eval(best_lgn_model, test_loader, mode=False)

diff_pct_points = (acc_soft - acc_hard) * 100.0

print(f"Accuracy (soft / before discretization): {acc_soft}")
print(f"Accuracy (hard / after discretization):  {acc_hard}")
print(f"Accuracy difference: {diff_pct_points:+.4f} percentage points")

print(f"Inference time (soft): {time_soft:.4f} s")
print(f"Inference time (hard): {time_hard:.4f} s")


Exists: True
Using runtime device: cpu
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
torch.int64 torch.int64
t

## Let's see the model srtucture

In [16]:
print(best_lgn_model)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): LogicLayer(784, 6000, train)
  (2): LogicLayer(6000, 6000, train)
  (3): LogicLayer(6000, 6000, train)
  (4): LogicLayer(6000, 6000, train)
  (5): GroupSum(k=10, tau=5.0)
)


####################################################################################################################################
### The next part is tested only on WSL Ubuntu

in this part we compile (discritize) the model and convert it into c and deploy it , which is much faster than the normal deployment

In [ ]:
from difflogic import CompiledLogicNet


compiled = CompiledLogicNet(best_lgn_model, device="cpu", num_bits=64)
compiled.compile(save_lib_path="/home/mahmo/compiled_logicnet.so")


In [ ]:

from difflogic import CompiledLogicNet


def compiled_eval(model, loader):
    with torch.no_grad():
        return float(np.mean([
            (model(
                x.round().bool().reshape(x.shape[0], -1).cpu()
            ).argmax(-1) == y.cpu()).to(torch.float32).mean().item()
            for x, y in loader
        ]))




compiled2 = CompiledLogicNet.load(
    "/mnt/f/difflogic codes/BooleanAlgebra__DeepLearning-main/trained_models/20260115_165626/compiled_logicnet.so",
    num_classes=best_lgn_model[-1].k,
    num_bits=64,
)

compiled_eval(compiled2, test_loader)


In [ ]:
# Print the logic gates used in each LogicLayer of the best LGN model 
import torch
from difflogic import LogicLayer
from difflogic.compiled_model import ALL_OPERATIONS

best_lgn_model.eval()

for layer_idx, layer in enumerate(best_lgn_model):
    if isinstance(layer, LogicLayer):
        gate_ids = layer.weights.argmax(1).detach().cpu().tolist()
        a_idx = layer.indices[0].detach().cpu().tolist()
        b_idx = layer.indices[1].detach().cpu().tolist()

        print(f"LogicLayer {layer_idx}:")
        for neuron_idx, (a, b, g) in enumerate(zip(a_idx, b_idx, gate_ids)):
            print(f"  neuron {neuron_idx}: inputs=({a},{b}) gate={ALL_OPERATIONS[g]}")


############################################################################################################################################################

## Generate a CSV file with the model gates and connections and JSON metadata for the model:

In [17]:
# Generate CSV + JSON metadata for the model
import csv
import json
import os
from difflogic import LogicLayer
from difflogic.compiled_model import ALL_OPERATIONS

# Example:
#path = r"trained_models/20260207_121201"
path= r"trained_models/20260309_224007"
#path= r"trained_models/20260309_220106"


def export_lgn_gates_csv_and_json(model, csv_path, json_path, args, class_count):
    # --- CSV ---
    rows = []
    for layer_idx, layer in enumerate(model):
        if isinstance(layer, LogicLayer):
            gate_ids = layer.weights.argmax(1).detach().cpu().tolist()
            a_idx = layer.indices[0].detach().cpu().tolist()
            b_idx = layer.indices[1].detach().cpu().tolist()

            for neuron_idx, (a, b, g) in enumerate(zip(a_idx, b_idx, gate_ids)):
                rows.append({
                    "layer": int(layer_idx),
                    "neuron": int(neuron_idx),
                    "input_a": int(a),
                    "input_b": int(b),
                    "gate_id": int(g),
                    "gate_name": str(ALL_OPERATIONS[g]),
                })

    with open(csv_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["layer", "neuron", "input_a", "input_b", "gate_id", "gate_name"],
        )
        writer.writeheader()
        writer.writerows(rows)

    # --- metadata derived from pixel mode ---
    pixel_mode = args.get("pixel_mode", "multi_threshold")
    num_niveis = int(args.get("num_niveis", 1))
    img_size = int(args["img_size"])

    if pixel_mode == "multi_threshold":
        in_dim = num_niveis * img_size * img_size
        binarization = {
            "type": "levels_thresholds",
            "num_niveis": num_niveis,
            "thresholds": [(i + 1) / float(num_niveis + 1) for i in range(num_niveis)],
            "comparison": ">",
        }
    elif pixel_mode == "threshold_05":
        in_dim = img_size * img_size
        binarization = {
            "type": "single_threshold",
            "threshold": 0.5,
            "comparison": ">",
        }
    else:
        raise ValueError(f"Unknown pixel_mode: {pixel_mode}")

    logic_layer_sizes = [int(layer.out_dim) for layer in model if isinstance(layer, LogicLayer)]

    meta = {
        "class_count": int(class_count),
        "img_size": img_size,
        "pixel_mode": pixel_mode,
        "num_niveis": num_niveis,
        "in_dim": int(in_dim),
        "num_layers": int(args["num_layers"]),
        "num_neurons": int(args["num_neurons"]),
        "tau": float(args["tau"]),
        "logic_layer_sizes": logic_layer_sizes,
        "gate_operations": list(ALL_OPERATIONS),
        "group_sum": {
            "type": "GroupSum",
            "class_count": int(class_count),
            "tau": float(args["tau"]),
        },
        "preprocess": {
            "to_tensor": True,
            "binarization": binarization,
            "flatten_order": "row_major",
        },
        "csv_fields": ["layer", "neuron", "input_a", "input_b", "gate_id", "gate_name"],
    }

    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)


In [19]:
# Example usage:
csv_path = os.path.join(path, "best_lgn_gates.csv")
json_path = os.path.join(path, "best_lgn_metadata.json")

export_lgn_gates_csv_and_json(best_lgn_model, csv_path, json_path, args, class_count=10)
print("CSV saved:", csv_path)
print("JSON saved:", json_path)


CSV saved: trained_models/20260309_224007/best_lgn_gates.csv
JSON saved: trained_models/20260309_224007/best_lgn_metadata.json


### Export Binarized Test CSV


In [20]:
# Export binarized test set to CSV for Rust inference
import os
import pandas as pd
import torch

out_path = os.path.join(path, "test_binarized.csv")

all_x = []
all_y = []

for x, y in test_loader:
    xb = x.round().bool().reshape(x.shape[0], -1).to(torch.int8).cpu()
    all_x.append(xb)
    all_y.append(y.cpu())

x_all = torch.cat(all_x, dim=0)
y_all = torch.cat(all_y, dim=0)

df = pd.DataFrame(x_all.numpy())
df.insert(0, "label", y_all.numpy())

df.to_csv(out_path, index=False)
print("Saved:", out_path)
print("Shape:", df.shape)  # rows = num test samples, cols = 1 + num_features


Saved: trained_models/20260309_224007/test_binarized.csv
Shape: (10000, 785)


In [21]:
# Just to check the generated CSV file

csv_path = os.path.join(path, "test_binarized.csv")
df = pd.read_csv(csv_path)

print("CSV path:", csv_path)
print("CSV shape:", df.shape)
print("First columns:", df.columns[:5].tolist())
print("Last columns:", df.columns[-5:].tolist())
print("Labels:", sorted(df["label"].unique().tolist()))

# Basic sanity checks
assert df.columns[0] == "label", "First column must be 'label'"
feature_vals = df.iloc[:, 1:].values
print("Feature min/max:", feature_vals.min(), feature_vals.max())
assert set(pd.unique(feature_vals.ravel())).issubset({0, 1}), "Features must be 0/1"

print("test_binarized.csv looks correct.")


CSV path: trained_models/20260309_224007/test_binarized.csv
CSV shape: (10000, 785)
First columns: ['label', '0', '1', '2', '3']
Last columns: ['779', '780', '781', '782', '783']
Labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Feature min/max: 0 1
test_binarized.csv looks correct.


Now let's test the generate csv file. To do so i will reconstruct the discrete model and test it.

In [22]:
# Reconstruct discrete model from CSV and test on the binarized test dataset
import pandas as pd
import torch

def _normalize_gate(name: str) -> str:
    s = str(name).strip().lower()
    for ch in [" ", "_", "-", "(", ")", ".", ","]:
        s = s.replace(ch, "")
    return s

def _gate_fn(name: str):
    key = _normalize_gate(name)
    aliases = {
        "zero": "zero", "0": "zero", "false": "zero",
        "one": "one", "1": "one", "true": "one",

        "a": "a", "b": "b",
        "nota": "not_a", "not_a": "not_a",
        "notb": "not_b", "not_b": "not_b",

        "and": "and",
        "or": "or",
        "xor": "xor",
        "notxor": "not_xor", "not_xor": "not_xor",

        "implies": "implies",
        "impliedby": "implied_by", "implied_by": "implied_by",

        "notand": "not_and", "not_and": "not_and",
        "notor": "not_or", "not_or": "not_or",

        "notimplies": "not_implies", "not_implies": "not_implies",
        "notimpliedby": "not_implied_by", "not_implied_by": "not_implied_by",
    }
    key = aliases.get(key, key)

    if key == "zero":
        return lambda a, b: torch.zeros_like(a, dtype=torch.bool)
    if key == "one":
        return lambda a, b: torch.ones_like(a, dtype=torch.bool)
    if key == "a":
        return lambda a, b: a
    if key == "b":
        return lambda a, b: b
    if key == "not_a":
        return lambda a, b: ~a
    if key == "not_b":
        return lambda a, b: ~b
    if key == "and":
        return lambda a, b: a & b
    if key == "or":
        return lambda a, b: a | b
    if key == "xor":
        return lambda a, b: a ^ b
    if key == "not_xor":
        return lambda a, b: ~(a ^ b)
    if key == "not_and":
        return lambda a, b: ~(a & b)
    if key == "not_or":
        return lambda a, b: ~(a | b)

    # implications
    if key == "implies":        # a -> b  ==  (~a) | b
        return lambda a, b: (~a) | b
    if key == "implied_by":     # a <- b  ==  a | (~b)
        return lambda a, b: a | (~b)
    if key == "not_implies":    # NOT(a -> b) == a & (~b)
        return lambda a, b: a & (~b)
    if key == "not_implied_by": # NOT(a <- b) == (~a) & b
        return lambda a, b: (~a) & b

    raise ValueError(f"Unknown gate name: {name}")


def load_discrete_layers_from_csv(csv_path):
    df = pd.read_csv(csv_path)
    layers = []
    for layer_id in sorted(df["layer"].unique()):
        layer_df = df[df["layer"] == layer_id].sort_values("neuron")
        layer = []
        for _, row in layer_df.iterrows():
            layer.append({
                "input_a": int(row["input_a"]),
                "input_b": int(row["input_b"]),
                "gate_name": row["gate_name"],
            })
        layers.append((layer_id, layer))
    return layers

def forward_discrete(layers, x_bool):
    x = x_bool
    for _, layer in layers:
        out = torch.empty((x.shape[0], len(layer)), dtype=torch.bool, device=x.device)
        for j, neuron in enumerate(layer):
            a = x[:, neuron["input_a"]]
            b = x[:, neuron["input_b"]]
            fn = _gate_fn(neuron["gate_name"])
            out[:, j] = fn(a, b)
        x = out
    return x

def eval_discrete_from_csv(csv_path, loader, class_count=10, device="cpu"):
    layers = load_discrete_layers_from_csv(csv_path)

    batch_accs = []
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device).round().bool().reshape(x.shape[0], -1)
            y = y.to(device)

            out = forward_discrete(layers, x)

            if out.shape[1] % class_count != 0:
                raise ValueError(
                    f"Last layer size ({out.shape[1]}) is not divisible by class_count ({class_count})."
                )

            logits = out.reshape(out.shape[0], class_count, -1).sum(dim=2)
            preds = logits.argmax(dim=1)

            acc_b = (preds == y).to(torch.float32).mean().item()
            batch_accs.append(acc_b)

    return float(np.mean(batch_accs))



In [23]:
# Example usage:
#path = 'trained_models/20260115_165626/'
csv_path = os.path.join(path, "best_lgn_gates.csv")
runtime_device = "cuda" if torch.cuda.is_available() else "cpu"
acc = eval_discrete_from_csv(csv_path, test_loader, class_count=10, device=runtime_device)
print(f"Discrete CSV model accuracy: {acc}")

Discrete CSV model accuracy: 0.9611000019311905


#############################################################################################################

## Now Let's test a RUST reconstruction for the model and see if it's working

In [ ]:
# Cell: create Rust project files (do NOT overwrite training `path`)
import pathlib

model_dir = pathlib.Path(path)          # current trained_models/... folder
proj = "lgn_eval"
proj_path = pathlib.Path(proj)
proj_path.mkdir(exist_ok=True)
(proj_path / "src").mkdir(exist_ok=True)

cargo_toml = r"""
[package]
name = "lgn_eval"
version = "0.1.0"
edition = "2021"

[dependencies]
csv = "1"
serde = { version = "1", features = ["derive"] }
serde_json = "1"
"""
(proj_path / "Cargo.toml").write_text(cargo_toml.strip() + "\n", encoding="utf-8")
print("Wrote", proj_path / "Cargo.toml")
print("Model dir:", model_dir)


In [ ]:
# Cell: write Rust code (base path passed from CLI, no hardcoded old path)
rust_code = r"""
use serde::Deserialize;
use std::collections::BTreeMap;
use std::error::Error;
use std::fs;
use std::path::PathBuf;

#[derive(Debug, Deserialize)]
struct GateRow {
    layer: usize,
    neuron: usize,
    input_a: usize,
    input_b: usize,
    gate_name: String,
}

#[derive(Debug, Deserialize)]
struct Meta {
    class_count: usize,
    in_dim: usize,
}

#[derive(Clone, Copy)]
enum Gate { Zero, One, A, B, NotA, NotB, And, Or, Xor, NotXor, NotAnd, NotOr, Implies, ImpliedBy, NotImplies, NotImpliedBy }

fn normalize_gate(name: &str) -> String {
    let mut s = name.trim().to_lowercase();
    for ch in [" ", "_", "-", "(", ")", ".", ","] { s = s.replace(ch, ""); }
    s
}

fn gate_from_name(name: &str) -> Result<Gate, String> {
    let key = match normalize_gate(name).as_str() {
        "zero" | "0" | "false" => "zero",
        "one" | "1" | "true" => "one",
        "a" => "a", "b" => "b",
        "nota" | "not_a" => "not_a",
        "notb" | "not_b" => "not_b",
        "and" => "and", "or" => "or", "xor" => "xor",
        "notxor" | "not_xor" => "not_xor",
        "implies" => "implies",
        "impliedby" | "implied_by" => "implied_by",
        "notand" | "not_and" => "not_and",
        "notor" | "not_or" => "not_or",
        "notimplies" | "not_implies" => "not_implies",
        "notimpliedby" | "not_implied_by" => "not_implied_by",
        _ => return Err(format!("Unknown gate name: {}", name)),
    };
    Ok(match key {
        "zero" => Gate::Zero, "one" => Gate::One, "a" => Gate::A, "b" => Gate::B,
        "not_a" => Gate::NotA, "not_b" => Gate::NotB, "and" => Gate::And, "or" => Gate::Or,
        "xor" => Gate::Xor, "not_xor" => Gate::NotXor, "not_and" => Gate::NotAnd, "not_or" => Gate::NotOr,
        "implies" => Gate::Implies, "implied_by" => Gate::ImpliedBy,
        "not_implies" => Gate::NotImplies, "not_implied_by" => Gate::NotImpliedBy,
        _ => return Err(format!("Unknown gate name: {}", name)),
    })
}

fn gate_eval(g: Gate, a: bool, b: bool) -> bool {
    match g {
        Gate::Zero => false, Gate::One => true, Gate::A => a, Gate::B => b, Gate::NotA => !a, Gate::NotB => !b,
        Gate::And => a & b, Gate::Or => a | b, Gate::Xor => a ^ b, Gate::NotXor => !(a ^ b),
        Gate::NotAnd => !(a & b), Gate::NotOr => !(a | b), Gate::Implies => (!a) | b, Gate::ImpliedBy => a | (!b),
        Gate::NotImplies => a & (!b), Gate::NotImpliedBy => (!a) & b,
    }
}

fn read_gates(path: &str) -> Result<Vec<Vec<(usize, usize, Gate)>>, Box<dyn Error>> {
    let mut rdr = csv::Reader::from_path(path)?;
    let mut by_layer: BTreeMap<usize, Vec<GateRow>> = BTreeMap::new();
    for res in rdr.deserialize() {
        let row: GateRow = res?;
        by_layer.entry(row.layer).or_default().push(row);
    }
    let mut layers = Vec::new();
    for (_lid, mut rows) in by_layer {
        rows.sort_by_key(|r| r.neuron);
        let mut layer = Vec::with_capacity(rows.len());
        for r in rows { layer.push((r.input_a, r.input_b, gate_from_name(&r.gate_name)?)); }
        layers.push(layer);
    }
    Ok(layers)
}

fn read_meta(path: &str) -> Result<Meta, Box<dyn Error>> {
    Ok(serde_json::from_str(&fs::read_to_string(path)?)?)
}

fn eval_layers(layers: &[Vec<(usize, usize, Gate)>], input: &[bool]) -> Vec<bool> {
    let mut x = input.to_vec();
    for layer in layers {
        let mut out = vec![false; layer.len()];
        for (j, (a, b, g)) in layer.iter().enumerate() { out[j] = gate_eval(*g, x[*a], x[*b]); }
        x = out;
    }
    x
}

fn main() -> Result<(), Box<dyn Error>> {
    let args: Vec<String> = std::env::args().collect();
    if args.len() != 2 {
        eprintln!("Usage: lgn_eval <model_dir>");
        std::process::exit(1);
    }
    let base = PathBuf::from(&args[1]);

    let gates_path = base.join("best_lgn_gates.csv");
    let meta_path  = base.join("best_lgn_metadata.json");
    let test_path  = base.join("test_binarized.csv");

    let meta = read_meta(meta_path.to_str().unwrap())?;
    let layers = read_gates(gates_path.to_str().unwrap())?;

    let mut rdr = csv::Reader::from_path(test_path)?;
    let mut correct = 0usize;
    let mut total = 0usize;

    for result in rdr.records() {
        let record = result?;
        let label: usize = record[0].parse()?;
        let mut input = vec![false; meta.in_dim];
        for i in 0..meta.in_dim {
            let v: u8 = record[i + 1].parse()?;
            input[i] = v != 0;
        }

        let out = eval_layers(&layers, &input);
        if out.len() % meta.class_count != 0 { return Err("Last layer size not divisible by class_count".into()); }

        let per_class = out.len() / meta.class_count;
        let mut best_c = 0usize;
        let mut best_sum = -1isize;
        for c in 0..meta.class_count {
            let mut sum = 0isize;
            for j in 0..per_class { if out[c * per_class + j] { sum += 1; } }
            if sum > best_sum { best_sum = sum; best_c = c; }
        }

        if best_c == label { correct += 1; }
        total += 1;
    }

    println!("Accuracy: {:.16}", correct as f64 / total as f64);
    Ok(())
}
"""
(pathlib.Path("lgn_eval/src/main.rs")).write_text(rust_code.strip() + "\n", encoding="utf-8")
print("Wrote lgn_eval/src/main.rs")


In [ ]:
# Cell: run
import os
cmd = f'cargo run --release --manifest-path lgn_eval/Cargo.toml -- "{os.fspath(model_dir)}"'
print(cmd)
!{cmd}


In [ ]:
model_dir = r"trained_models/20260207_121201"
!cargo run --release --manifest-path lgn_eval/Cargo.toml -- "{model_dir}"
